# Tostal — Train & Evaluate All Models

This notebook clones (or pulls) the repo, installs dependencies, and runs:
1. **Facies classifier** — XGBoost + spatial features on FORCE 2020 well logs
2. **DINOv2 classifier** — vit_s14 backbone on DCID drill core images
3. **Evaluate** — accuracy, F1, confusion matrix, top-k

Estimated runtime: ~3 min CPU (facies) + ~3 min GPU (DINOv2)

In [ ]:
import os, sys, subprocess
from pathlib import Path

## 1. Clone or Pull the Repository

In [ ]:
REPO_URL = "https://github.com/beaglabs/tostal.git"

# Detect if we're already inside the repo
repo_root = Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists():
        break
    repo_root = repo_root.parent

if (repo_root / ".git").exists():
    print(f"Found existing repo at {repo_root}, pulling latest...")
    subprocess.run(["git", "pull"], cwd=repo_root, check=True)
else:
    repo_root = Path.home() / "tostal"
    if (repo_root / ".git").exists():
        print(f"Repo already exists at {repo_root}, pulling latest...")
        subprocess.run(["git", "pull"], cwd=repo_root, check=True)
    elif repo_root.exists():
        print(f"Directory {repo_root} exists but is not a git repo, removing...")
        import shutil
        shutil.rmtree(repo_root)
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    else:
        print(f"Cloning into {repo_root}...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)

print(f"Repo root: {repo_root}")

## 2. Install Dependencies

In [ ]:
requirements = repo_root / "api" / "requirements.txt"
if requirements.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements)], check=True)
else:
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "xgboost", "pykrige", "scikit-learn", "torch", "torchvision",
        "scipy", "datasets", "lasio", "openpyxl", "tqdm", "pillow",
    ], check=True)

print("Dependencies installed.")

## 3. Train Facies Classifier (XGBoost + Spatial Features)

Downloads FORCE 2020 well logs, engineers spatial features (normalized depth,
rolling-window stats, neighboring values), and trains an XGBoost classifier.
Saves to `training/models/xgboost_facies.json`.

In [ ]:
%cd {repo_root}

# Clean stale cache so process_force2020 does fresh extraction
import shutil
data_dir = Path("data/force2020")
processed_pt = data_dir / "processed.pt"
las_dir = data_dir / "las"
if processed_pt.exists():
    processed_pt.unlink()
if las_dir.exists():
    shutil.rmtree(las_dir)

subprocess.run([sys.executable, "training/scripts/train_xgboost_facies.py"], check=True)

## 4. Train DINOv2 Classifier (Vit-S/14)

Loads 1000 DCID drill core images, extracts features with frozen DINOv2 vit_s14
backbone, and trains a lightweight classifier head. Saves to `training/models/dino_classifier.pt`.

**~2 min on GPU, ~10 min on CPU.**

In [ ]:
%cd {repo_root}
subprocess.run([sys.executable, "training/scripts/train_dino_classifier.py"], check=True)

## 5. Evaluate Models

XGBoost: accuracy, F1, per-class F1, confusion matrix on full FORCE 2020.
DINOv2: top-1 and top-3 accuracy on held-out DCID subset.

In [ ]:
%cd {repo_root}
subprocess.run([sys.executable, "training/scripts/evaluate.py"], check=True)

## 6. Verify Outputs

In [ ]:
models_dir = repo_root / "training" / "models"
for f in sorted(models_dir.glob("*")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:40s}  {size_mb:8.2f} MB")
print("\nDone.")